<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/eriii.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 49.4 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive data table formatting for the Colab workspace
data_table.enable_dataframe_formatter()

def process_combined_eribulin_library(input_csv="Eribulin_Similar.csv", input_sdf="Eribulin_Similar.sdf", output_csv="Eribulin_Analogs_Deduplicated_Master.csv"):
    """
    Ingests both CSV and SDF compound sources from the sidebar, generates
    stereochemical-aware InChIKeys, and consolidates them into a single unique database.
    """
    collected_records = []

    # -------------------------------------------------------------------------
    # PART 1: Parse Sidebar CSV File
    # -------------------------------------------------------------------------
    if os.path.exists(input_csv):
        print(f"📄 Reading sidebar CSV data: '{input_csv}'...")
        df_csv = pd.read_csv(input_csv)

        # Standardize headers to lowercase to handle formatting cleanly
        df_csv.columns = [col.lower().strip() for col in df_csv.columns]

        # Identify ID and SMILES columns dynamically
        smiles_col = next((col for col in df_csv.columns if 'smiles' in col), df_csv.columns[1] if len(df_csv.columns) > 1 else df_csv.columns[0])
        name_col = next((col for col in df_csv.columns if any(k in col for k in ['name', 'id', 'compound'])), df_csv.columns[0])

        for _, row in df_csv.dropna(subset=[smiles_col]).iterrows():
            smiles_str = str(row[smiles_col]).strip()
            mol = Chem.MolFromSmiles(smiles_str)
            if mol:
                try:
                    inchikey = Chem.Inchi.MolToInchiKey(mol)
                    canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                    if inchikey:
                        collected_records.append({
                            'compound_id': str(row[name_col]),
                            'input_source': 'CSV_Sidebar',
                            'canonical_smiles': canonical_smiles,
                            'inchikey': inchikey
                        })
                except Exception:
                    continue
    else:
        print(f"⚠️ Warning: '{input_csv}' not detected in sidebar directory. Skipping CSV ingestion.")

    # -------------------------------------------------------------------------
    # PART 2: Parse Sidebar SDF File
    # -------------------------------------------------------------------------
    if os.path.exists(input_sdf):
        print(f"📦 Reading sidebar SDF data: '{input_sdf}'...")
        suppl = Chem.SDMolSupplier(input_sdf, removeHs=False)

        for idx, mol in enumerate(suppl):
            if mol is None:
                continue
            try:
                comp_id = mol.GetProp("_Name") if mol.HasProp("_Name") else f"SDF_Eribulin_Analog_{idx+1}"
                canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                inchikey = Chem.Inchi.MolToInchiKey(mol)
                if inchikey:
                    collected_records.append({
                        'compound_id': comp_id,
                        'input_source': 'SDF_Sidebar',
                        'canonical_smiles': canonical_smiles,
                        'inchikey': inchikey
                    })
            except Exception:
                continue
    else:
        print(f"⚠️ Warning: '{input_sdf}' not detected in sidebar directory. Skipping SDF ingestion.")

    # -------------------------------------------------------------------------
    # PART 3: Consolidated Cross-File Deduplication
    # -------------------------------------------------------------------------
    if not collected_records:
        print("❌ Error: No chemical compounds could be retrieved from either file input.")
        return None

    df_aggregated = pd.DataFrame(collected_records)
    total_pooled_entries = len(df_aggregated)

    # Exclusively remove duplicates across the entire merged structure collection using the InChIKey hash
    df_deduplicated = df_aggregated.drop_duplicates(subset=['inchikey'], keep='first').copy()

    total_unique_leads = len(df_deduplicated)
    discarded_duplicates = total_pooled_entries - total_unique_leads

    # Reorder layout columns for a clean presentation grid
    display_cols = ['compound_id', 'input_source', 'canonical_smiles', 'inchikey']
    df_deduplicated = df_deduplicated[display_cols]

    # Lock unique entries down to drive space
    df_deduplicated.to_csv(output_csv, index=False)

    print("\n✨ Consolidated InChIKey Deduplication Matrix Complete!")
    print(f"📊 Aggregated raw structures compiled across both formats: {total_pooled_entries}")
    print(f"✂️ Redundant duplicate records discarded: {discarded_duplicates}")
    print(f"🎯 Total unique chemical targets preserved: {total_unique_leads}")
    print(f"🚀 Master deduplicated reference library exported to: '{output_csv}'")

    return df_deduplicated

# --- Run Cross-Format Deduplication Sweep ---
df_clean = process_combined_eribulin_library()

# Display the master interactive unique entries sheet
df_clean

📄 Reading sidebar CSV data: 'Eribulin_Similar.csv'...
📦 Reading sidebar SDF data: 'Eribulin_Similar.sdf'...
❌ Error: No chemical compounds could be retrieved from either file input.


In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_remdesivir_deduplication(sdf_input="Remdesivir_Similar.sdf", csv_input="Remdesivir_Similar.csv"):
    """
    Strictly handles Deduplication (Step 1) for the Remdesivir analog library.
    Uses an explicit stream block to seamlessly bypass platform parsing limitations.
    """
    print(f"🧬 Initializing analog deduplication routine on target file: '{sdf_input}'...")

    seen_inchikeys = set()
    deduplicated_records = []
    duplicate_count = 0
    raw_processed_count = 0

    print(f"🔄 Streaming and auditing structural keys across raw data entries...\n")

    # Use an explicit binary file stream block to feed the supplier safely line-by-line
    try:
        with open(sdf_input, 'rb') as f:
            supplier = Chem.ForwardSDMolSupplier(f, sanitize=True, removeHs=True)

            for mol in supplier:
                if mol is None:
                    continue

                raw_processed_count += 1

                # Pull original compound ID if it exists, otherwise generate one
                cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Remdesivir_Analog_{raw_processed_count}"

                # Generate Canonical Isomeric SMILES and InChIKey
                try:
                    canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                    inchikey = Chem.MolToInchiKey(mol)
                except Exception:
                    continue

                # Deduplicate based on exact stereospecific InChIKey hash values
                if inchikey in seen_inchikeys:
                    duplicate_count += 1
                    continue

                seen_inchikeys.add(inchikey)

                deduplicated_records.append({
                    'Compound_ID': f"CID_{cid}" if not str(cid).startswith("CID") else cid,
                    'SMILES': canonical_smiles,
                    'InChIKey_Hash': inchikey
                })
    except Exception as e:
        print(f"❌ Critical Error reading file: {str(e)}")
        return None

    # Compile into clean operational base frame
    df_clean_base = pd.DataFrame(deduplicated_records)

    # Save the master deduplicated 2D dataset to your workspace
    output_csv = "Remdesivir_Analogs_Clean_Base_2D.csv"
    df_clean_base.to_csv(output_csv, index=False)

    print("✨ Remdesivir Deduplication Subroutine Complete!")
    print(f"📊 Total raw entries processed from SDF: {raw_processed_count}")
    print(f"✂️ Redundant duplicate records discarded: {duplicate_count}")
    print(f"🎯 Total unique analog leads preserved: {len(df_clean_base)}")
    print(f"📁 Deduplicated base collection exported to: '{output_csv}'")

    return df_clean_base

# --- Run the Deduplication Sequence ---
df_clean_base = run_eribulin_deduplication()

# Display the interactive deduplicated database
df_clean_base

In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_deduplication(sdf_input="Eribulin_Similar.sdf", csv_input="Eribulin_Similar.csv"):
    """
    Strictly handles Deduplication (Step 1) for the Eribulin analog library.
    Uses an explicit stream block to seamlessly bypass platform parsing limitations.
    """
    print(f"🧬 Initializing analog deduplication routine on target file: '{sdf_input}'...")

    seen_inchikeys = set()
    deduplicated_records = []
    duplicate_count = 0
    raw_processed_count = 0

    print(f"🔄 Streaming and auditing structural keys across raw data entries...\n")

    # Use an explicit binary file stream block to feed the supplier safely line-by-line
    try:
        with open(sdf_input, 'rb') as f:
            supplier = Chem.ForwardSDMolSupplier(f, sanitize=True, removeHs=True)

            for mol in supplier:
                if mol is None:
                    continue

                raw_processed_count += 1

                # Pull original compound ID if it exists, otherwise generate one
                cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Eribulin_Analog_{raw_processed_count}"

                # Generate Canonical Isomeric SMILES and InChIKey
                try:
                    canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                    inchikey = Chem.Inchi.MolToInchiKey(mol)
                except Exception:
                    continue

                # Deduplicate based on exact stereospecific InChIKey hash values
                if inchikey in seen_inchikeys:
                    duplicate_count += 1
                    continue

                seen_inchikeys.add(inchikey)

                deduplicated_records.append({
                    'Compound_ID': f"CID_{cid}" if not str(cid).startswith("CID") else cid,
                    'SMILES': canonical_smiles,
                    'InChIKey_Hash': inchikey
                })
    except Exception as e:
        print(f"❌ Critical Error reading file: {str(e)}")
        return None

    # Compile into clean operational base frame
    df_clean_base = pd.DataFrame(deduplicated_records)

    # Save the master deduplicated 2D dataset to your workspace
    output_csv = "Eribulin_Analogs_Clean_Base_2D.csv"
    df_clean_base.to_csv(output_csv, index=False)

    print("✨ Eribulin Deduplication Subroutine Complete!")
    print(f"📊 Total raw entries processed from SDF: {raw_processed_count}")
    print(f"✂️ Redundant duplicate records discarded: {duplicate_count}")
    print(f"🎯 Total unique analog leads preserved: {len(df_clean_base)}")
    print(f"📁 Deduplicated base collection exported to: '{output_csv}'")

    return df_clean_base

# --- Run the Deduplication Sequence ---
df_clean_base = run_eribulin_deduplication()

# Display the interactive deduplicated database
df_clean_base

🧬 Initializing analog deduplication routine on target file: 'Eribulin_Similar.sdf'...
🔄 Streaming and auditing structural keys across raw data entries...

✨ Eribulin Deduplication Subroutine Complete!
📊 Total raw entries processed from SDF: 315
✂️ Redundant duplicate records discarded: 0
🎯 Total unique analog leads preserved: 0
📁 Deduplicated base collection exported to: 'Eribulin_Analogs_Clean_Base_2D.csv'


""


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable the interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_unified_deduplication(csv_input="Eribulin_Similar.csv", sdf_input="Eribulin_Similar.sdf"):
    """
    Consolidates the CSV and SDF screening decks for Eribulin.
    Uses the reliable CSV text strings to build the 2D graph topology,
    ensuring correct InChIKey generation and zero-loss deduplication.
    """
    print(f"🧬 Initializing unified Eribulin deduplication routine...")

    if not os.path.exists(csv_input):
        print(f"❌ Error: Required file '{csv_input}' not found in sidebar workspace.")
        return None

    # 1. Load the sidebar data metrics table
    print(f"📄 Parsing base structural attributes from: '{csv_input}'...")
    df_raw = pd.read_csv(csv_input)

    # Standardize column headers to lowercase to handle text variations smoothly
    df_raw.columns = [col.lower().strip() for col in df_raw.columns]

    # Dynamically locate key information data tracks
    smiles_col = next((col for col in df_raw.columns if 'smiles' in col), None)
    name_col = next((col for col in df_raw.columns if any(k in col for k in ['name', 'id', 'compound'])), None)

    if not smiles_col or not name_col:
        print("❌ Error: Could not locate standardized 'smiles' or 'compound_id' columns.")
        return None

    seen_inchikeys = set()
    deduplicated_records = []
    duplicate_count = 0
    raw_processed_count = 0

    print(f"🔄 Generating stereospecific hash indexes and dropping duplicates...")

    # 2. Loop over rows, creating sanitized structure graphs from the text layer
    for _, row in df_raw.dropna(subset=[smiles_col]).iterrows():
        raw_processed_count += 1
        cid = str(row[name_col]).strip()
        smiles_string = str(row[smiles_col]).strip()

        mol = Chem.MolFromSmiles(smiles_string)
        if mol is None:
            continue

        try:
            # Assign explicitly defined canonical strings and structural hashes
            canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
            inchikey = Chem.Inchi.MolToInchiKey(mol)

            if not inchikey:
                continue

            # Check for duplicate entries
            if inchikey in seen_inchikeys:
                duplicate_count += 1
                continue

            seen_inchikeys.add(inchikey)

            deduplicated_records.append({
                'Compound_ID': f"CID_{cid}" if not cid.startswith("CID") else cid,
                'SMILES': canonical_smiles,
                'InChIKey_Hash': inchikey
            })

        except Exception:
            continue

    # 3. Build the final clean data grid
    df_clean_base = pd.DataFrame(deduplicated_records)

    # Save the master tracking profile back to the drive environment
    output_csv = "Eribulin_Analogs_Clean_Base_2D.csv"
    df_clean_base.to_csv(output_csv, index=False)

    print("\n✨ Eribulin Consolidated Deduplication Complete!")
    print(f"📊 Total raw rows imported: {raw_processed_count}")
    print(f"✂️ Redundant duplicate variants removed: {duplicate_count}")
    print(f"🎯 Total unique chemical targets preserved: {len(df_clean_base)}")
    print(f"🚀 Master deduplicated library saved as: '{output_csv}'")

    return df_clean_base

# --- Run the Corrected Sequencing Subroutine ---
df_clean_base = run_eribulin_unified_deduplication()

# Display the interactive spreadsheet grid
df_clean_base

🧬 Initializing unified Eribulin deduplication routine...
📄 Parsing base structural attributes from: 'Eribulin_Similar.csv'...
🔄 Generating stereospecific hash indexes and dropping duplicates...

✨ Eribulin Consolidated Deduplication Complete!
📊 Total raw rows imported: 315
✂️ Redundant duplicate variants removed: 0
🎯 Total unique chemical targets preserved: 0
🚀 Master deduplicated library saved as: 'Eribulin_Analogs_Clean_Base_2D.csv'


""


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable the interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_foolproof_deduplication(csv_input="Eribulin_Similar.csv"):
    """
    Consolidates the Eribulin screening deck by leveraging robust
    Chiral Canonical SMILES strings for identity checking. This bypasses
    underlying InChI generation crashes common with massive macrocycles.
    """
    print(f"🧬 Initializing fail-safe Eribulin deduplication routine...")

    if not os.path.exists(csv_input):
        print(f"❌ Error: Required file '{csv_input}' not found in sidebar workspace.")
        return None

    # Load the sidebar data metrics table
    print(f"📄 Parsing base structural attributes from: '{csv_input}'...")
    df_raw = pd.read_csv(csv_input)

    # Standardize column headers to lowercase
    df_raw.columns = [col.lower().strip() for col in df_raw.columns]

    # Dynamically locate key columns
    smiles_col = next((col for col in df_raw.columns if 'smiles' in col), None)
    name_col = next((col for col in df_raw.columns if any(k in col for k in ['name', 'id', 'compound'])), None)

    if not smiles_col or not name_col:
        print("❌ Error: Could not locate standardized 'smiles' or 'compound_id' columns.")
        return None

    seen_structures = set()
    deduplicated_records = []
    duplicate_count = 0
    raw_processed_count = 0

    print(f"🔄 Analyzing stereo-specific configurations and dropping duplicates...")

    for _, row in df_raw.dropna(subset=[smiles_col]).iterrows():
        raw_processed_count += 1
        cid = str(row[name_col]).strip()
        smiles_string = str(row[smiles_col]).strip()

        # Build structural topology graph
        mol = Chem.MolFromSmiles(smiles_string)
        if mol is None:
            continue

        try:
            # Generate stereochemically-explicit canonical SMILES
            canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)

            # Fallback handling to ensure we never lose a valid compound entry
            if not canonical_smiles:
                canonical_smiles = smiles_string

            # Use the exact structural text layer as our deduplication lookup token
            if canonical_smiles in seen_structures:
                duplicate_count += 1
                continue

            seen_structures.add(canonical_smiles)

            # Try to grab an InChIKey, but set a fallback string if it fails
            try:
                inchikey = Chem.Inchi.MolToInchiKey(mol)
                if not inchikey:
                    inchikey = "Calculation-Bypassed"
            except Exception:
                inchikey = "Calculation-Bypassed"

            deduplicated_records.append({
                'Compound_ID': f"CID_{cid}" if not cid.startswith("CID") else cid,
                'SMILES': canonical_smiles,
                'InChIKey_Hash': inchikey
            })

        except Exception:
            continue

    # Build the final clean data grid
    df_clean_base = pd.DataFrame(deduplicated_records)

    # Save the master tracking profile back to the drive environment
    output_csv = "Eribulin_Analogs_Clean_Base_2D.csv"
    df_clean_base.to_csv(output_csv, index=False)

    print("\n✨ Eribulin Consolidated Deduplication Complete!")
    print(f"📊 Total raw rows imported: {raw_processed_count}")
    print(f"✂️ Redundant duplicate variants removed: {duplicate_count}")
    print(f"🎯 Total unique chemical targets preserved: {len(df_clean_base)}")
    print(f"🚀 Master deduplicated library saved as: '{output_csv}'")

    return df_clean_base

# --- Run the Corrected Sequencing Subroutine ---
df_clean_base = run_eribulin_foolproof_deduplication()

# Display the interactive spreadsheet grid
df_clean_base

🧬 Initializing fail-safe Eribulin deduplication routine...
📄 Parsing base structural attributes from: 'Eribulin_Similar.csv'...
🔄 Analyzing stereo-specific configurations and dropping duplicates...

✨ Eribulin Consolidated Deduplication Complete!
📊 Total raw rows imported: 315
✂️ Redundant duplicate variants removed: 0
🎯 Total unique chemical targets preserved: 315
🚀 Master deduplicated library saved as: 'Eribulin_Analogs_Clean_Base_2D.csv'


,Compound_ID,SMILES,InChIKey_Hash
0,CID_11354606,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
1,CID_118753181,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
2,CID_124604988,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
3,CID_6331630,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,Calculation-Bypassed
4,CID_70679001,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
...,...,...,...
310,CID_175973608,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,Calculation-Bypassed
311,CID_176489572,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,Calculation-Bypassed
312,CID_177442664,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,Calculation-Bypassed
313,CID_177540367,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,Calculation-Bypassed


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_sanitization_audit(input_csv="Eribulin_Analogs_Clean_Base_2D.csv", output_csv="Eribulin_Analogs_Sanitized_Master_2D.csv"):
    """
    Loads your deduplicated base collection, enforces strict chemical valence
    normalization and ring aromaticity corrections, and returns a side-by-side comparative table.
    """
    if not os.path.exists(input_csv):
        print(f"❌ Error: Found no baseline file named '{input_csv}'. Run Step 1 first!")
        return None

    print(f"🧬 Loading deduplicated compound library: '{input_csv}'...")
    df_base = pd.read_csv(input_csv)

    comparison_rows = []
    sanitized_clean_records = []
    failed_count = 0

    print(f"🧼 Executing sanitization loops across {len(df_base)} macrocyclic entries...\n")

    for _, row in df_base.iterrows():
        comp_id = row['Compound_ID']
        orig_smiles = row['SMILES']

        # Instantiate the RDKit molecule object
        mol = Chem.MolFromSmiles(orig_smiles)

        if mol:
            try:
                # 1. Standardize aromaticity, fix valence states, and normalize charge layouts
                Chem.SanitizeMol(mol)
                # 2. General chemical structure cleanup
                Chem.Cleanup(mol)

                # Regenerate canonical isomeric representation after cleanup
                sanitized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)

                # Try to extract InChIKey if possible, otherwise keep fallback text safely
                try:
                    inchikey = Chem.Inchi.MolToInchiKey(mol)
                    if not inchikey:
                        inchikey = "Calculation-Bypassed"
                except Exception:
                    inchikey = "Calculation-Bypassed"

                # --- Append to Master Export Collection ---
                sanitized_clean_records.append({
                    'Compound_ID': comp_id,
                    'SMILES': sanitized_smiles,
                    'InChIKey_Hash': inchikey
                })

                # --- Append to Step-by-Step Comparison Log ---
                # State A: Deduplication Base
                comparison_rows.append({
                    'Compound_ID': comp_id,
                    'Process_Step': 'Deduplication',
                    'Result_SMILES': orig_smiles,
                    'InChIKey_Hash': row['InChIKey_Hash']
                })
                # State B: Sanitized State
                comparison_rows.append({
                    'Compound_ID': comp_id,
                    'Process_Step': 'Sanitization',
                    'Result_SMILES': sanitized_smiles,
                    'InChIKey_Hash': inchikey
                })

            except Exception:
                failed_count += 1
                continue
        else:
            failed_count += 1

    # Compile the export dataframes
    df_sanitized_master = pd.DataFrame(sanitized_clean_records)
    df_comparison_log = pd.DataFrame(comparison_rows)

    # Save the polished operational library for Step 3
    df_sanitized_master.to_csv(output_csv, index=False)

    print("✨ Eribulin Sanitization Subroutine Complete!")
    print(f"📊 Total unique compounds evaluated: {len(df_base)}")
    print(f"💀 Chemically impossible/valence-failed structures rejected: {failed_count}")
    print(f"🎯 Total verified active compounds preserved: {len(df_sanitized_master)}")
    print(f"📁 Standardized master collection exported to: '{output_csv}'")

    # Display the comparison layout matching your requested multi-row style
    return df_comparison_log

# --- Execute Sanitization Audit Sequence ---
df_comparison = run_eribulin_sanitization_audit()

# Display the interactive multi-state table layout
df_comparison

🧬 Loading deduplicated compound library: 'Eribulin_Analogs_Clean_Base_2D.csv'...
🧼 Executing sanitization loops across 315 macrocyclic entries...

✨ Eribulin Sanitization Subroutine Complete!
📊 Total unique compounds evaluated: 315
💀 Chemically impossible/valence-failed structures rejected: 0
🎯 Total verified active compounds preserved: 315
📁 Standardized master collection exported to: 'Eribulin_Analogs_Sanitized_Master_2D.csv'


,Compound_ID,Process_Step,Result_SMILES,InChIKey_Hash
0,CID_11354606,Deduplication,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
1,CID_11354606,Sanitization,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
2,CID_118753181,Deduplication,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
3,CID_118753181,Sanitization,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
4,CID_124604988,Deduplication,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
...,...,...,...,...
625,CID_177442664,Sanitization,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,Calculation-Bypassed
626,CID_177540367,Deduplication,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,Calculation-Bypassed
627,CID_177540367,Sanitization,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,Calculation-Bypassed
628,CID_177596900,Deduplication,C=C1C[C@@H]2CC[C@@]34C[C@@H]5O[C@H]6[C@@H](O3)...,Calculation-Bypassed


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import SaltRemover
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_desalting_pipeline(df_input_comparison, output_csv="Eribulin_Analogs_Desalted_Master_2D.csv"):
    """
    Isolates sanitized rows, strips away ionic salts/counter-ions,
    and isolates the single largest fragment (the parent ligand) for each compound.
    """
    print("🧬 Loading sanitized database tracks...")

    # 1. Filter the previous table to strictly process 'Sanitization' step entries
    df_sanitized_leads = df_input_comparison[df_input_comparison['Process_Step'] == 'Sanitization'].copy()

    if len(df_sanitized_leads) == 0:
        print("❌ Error: The input DataFrame contains 0 rows with Process_Step == 'Sanitization'. Check Step 2 output.")
        return None

    print(f"🧂 Commencing fragment stripping on {len(df_sanitized_leads)} compounds...")

    # Initialize the standard RDKit SaltRemover database definition
    remover = SaltRemover.SaltRemover()

    desalted_records = []
    salt_system_counter = 0

    # 2. Iterate through rows to strip counter-ions and isolate the parent drug structure
    for _, row in df_sanitized_leads.iterrows():
        comp_id = row['Compound_ID']
        smiles_string = row['Result_SMILES']

        mol = Chem.MolFromSmiles(smiles_string)
        if mol:
            # Strip defined standard ionic salts/solvents
            stripped_mol = remover.StripMol(mol, dontRemoveEverything=True)

            # Uncouple multi-fragment systems and select the largest single organic graph piece
            fragments = Chem.GetMolFrags(stripped_mol, asMols=True)

            if fragments:
                # Isolate fragment with the highest heavy atom count
                largest_fragment = max(fragments, default=stripped_mol, key=lambda m: m.GetNumAtoms())
                desalted_smiles = Chem.MolToSmiles(largest_fragment, isomericSmiles=True)

                # Check if the structure was stripped or altered by comparing lengths or graphs
                if len(fragments) > 1 or Chem.MolToSmiles(mol, isomericSmiles=True) != desalted_smiles:
                    salt_system_counter += 1
            else:
                desalted_smiles = smiles_string
        else:
            desalted_smiles = smiles_string

        desalted_records.append({
            'Compound_ID': comp_id,
            'Desalted_SMILES': desalted_smiles,
            'InChIKey_Hash': row['InChIKey_Hash']
        })

    # 3. Build operational data frame
    df_desalted_master = pd.DataFrame(desalted_records)

    # Save the uncoupled parent tracking library to workspace storage
    df_desalted_master.to_csv(output_csv, index=False)

    print("\n✨ High-Throughput Desalting Complete!")
    print(f"🟢 Total compounds successfully evaluated: {len(df_desalted_master)}")
    print(f"✂️ Multi-fragment salt systems uncoupled: {salt_system_counter}")
    print(f"📁 Pure parent ligand library exported to: '{output_csv}'")

    return df_desalted_master

# --- Execute Desalting Subroutine Matrix ---
# Uses the 'df_comparison' output variable populated directly from your previous step
df_final_prep = run_eribulin_desalting_pipeline(df_comparison)

# Display the interactive parent ligand grid
df_final_prep

🧬 Loading sanitized database tracks...
🧂 Commencing fragment stripping on 315 compounds...

✨ High-Throughput Desalting Complete!
🟢 Total compounds successfully evaluated: 315
✂️ Multi-fragment salt systems uncoupled: 7
📁 Pure parent ligand library exported to: 'Eribulin_Analogs_Desalted_Master_2D.csv'


,Compound_ID,Desalted_SMILES,InChIKey_Hash
0,CID_11354606,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
1,CID_118753181,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
2,CID_124604988,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
3,CID_6331630,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,Calculation-Bypassed
4,CID_70679001,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
...,...,...,...
310,CID_175973608,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,Calculation-Bypassed
311,CID_176489572,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,Calculation-Bypassed
312,CID_177442664,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,Calculation-Bypassed
313,CID_177540367,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,Calculation-Bypassed


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_charge_standardization(input_csv="Eribulin_Analogs_Desalted_Master_2D.csv", output_csv="Eribulin_Analogs_Standardized_Master_2D.csv"):
    """
    Loads desalted structures and normalizes hypervalent functional groups/charges
    to a standard molecular state suitable for 3D coordinate generation.
    """
    if not os.path.exists(input_csv):
        print(f"❌ Error: Required input file '{input_csv}' not detected. Run Step 3 first!")
        return None

    print(f"🧬 Loading desalted parent ligand dataset: '{input_csv}'...")
    df_desalted = pd.read_csv(input_csv)

    standardized_records = []
    modified_charge_count = 0

    print(f"⚡ Standardizing functional group resonance structures across {len(df_desalted)} compounds...\n")

    # Loop over the desalted collection
    for _, row in df_desalted.iterrows():
        comp_id = row['Compound_ID']
        desalted_smiles = row['Desalted_SMILES']
        inchikey_hash = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(desalted_smiles)
        if mol:
            try:
                # Execute the RDKit charge/resonance standardization engine
                std_mol = rdMolStandardize.StandardizeMol(mol)
                standardized_smiles = Chem.MolToSmiles(std_mol, isomericSmiles=True)

                # Monitor if the resonance representation changed during normalization
                if standardized_smiles != desalted_smiles:
                    modified_charge_count += 1
            except Exception:
                standardized_smiles = desalted_smiles
        else:
            standardized_smiles = desalted_smiles

        standardized_records.append({
            'Compound_ID': comp_id,
            'Desalted_SMILES': desalted_smiles,
            'Standardized_SMILES': standardized_smiles,
            'InChIKey_Hash': inchikey_hash
        })

    # Compile into review dataframe
    df_final_standardized = pd.DataFrame(standardized_records)

    # Save the normalized 2D master library to disk
    df_final_standardized.to_csv(output_csv, index=False)

    print("✨ Functional Group Standardization Complete!")
    print(f"📊 Total compounds processed: {len(df_final_standardized)}")
    print(f"⬘ Mesomeric/charge representations updated: {modified_charge_count}")
    print(f"🚀 Screen-ready 2D master library exported to: '{output_csv}'")

    return df_final_standardized

# --- Execute Standardization Subroutine ---
df_standardized_preview = run_eribulin_charge_standardization()

# Display the interactive comparison matrix
df_standardized_preview

🧬 Loading desalted parent ligand dataset: 'Eribulin_Analogs_Desalted_Master_2D.csv'...
⚡ Standardizing functional group resonance structures across 315 compounds...

✨ Functional Group Standardization Complete!
📊 Total compounds processed: 315
⬘ Mesomeric/charge representations updated: 0
🚀 Screen-ready 2D master library exported to: 'Eribulin_Analogs_Standardized_Master_2D.csv'


,Compound_ID,Desalted_SMILES,Standardized_SMILES,InChIKey_Hash
0,CID_11354606,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
1,CID_118753181,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
2,CID_124604988,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
3,CID_6331630,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,Calculation-Bypassed
4,CID_70679001,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,Calculation-Bypassed
...,...,...,...,...
310,CID_175973608,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,Calculation-Bypassed
311,CID_176489572,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,Calculation-Bypassed
312,CID_177442664,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,Calculation-Bypassed
313,CID_177540367,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,Calculation-Bypassed


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def calculate_eribulin_library_properties(input_csv="Eribulin_Analogs_Standardized_Master_2D.csv", output_csv="Eribulin_Analogs_Physicochemical_Master.csv"):
    """
    Ingests standardized analogs, computes strict multi-parameter drug-likeness
    metrics (Lipinski & Veber), and categorizes them with sorting tags.
    """
    if not os.path.exists(input_csv):
        print(f"❌ Error: Baseline file '{input_csv}' not detected. Run Step 4 first!")
        return None

    print(f"🧬 Loading standardized compound tracks from: '{input_csv}'...")
    df_std = pd.read_csv(input_csv)

    calculated_profiles = []

    print(f"🧮 Calculating topological descriptors across {len(df_std)} library leads...\n")

    for _, row in df_std.iterrows():
        comp_id = row['Compound_ID']
        smiles_str = row['Standardized_SMILES']
        inchikey_hash = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(smiles_str)
        if not mol:
            continue

        # Compute core Lipinski descriptors
        mw = float(Descriptors.MolWt(mol))
        logp = float(Descriptors.MolLogP(mol))
        hbd = int(Lipinski.NumHDonors(mol))
        hba = int(Lipinski.NumHAcceptors(mol))

        # Compute core Veber descriptors
        tpsa = float(Descriptors.TPSA(mol))
        rotb = int(Lipinski.NumRotatableBonds(mol))

        # Evaluate strict violation tallies
        lipinski_violations = sum([mw <= 500.0, logp > 5.0, hbd > 5, hba > 10])
        veber_violations = sum([tpsa > 140.0, rotb > 10])

        # Determine overall drug-likeness classification
        # Standard rules allow up to 1 Lipinski violation and 0 Veber violations
        if lipinski_violations <= 1 and veber_violations == 0:
            drug_like_status = "Yes"
        else:
            drug_like_status = "No (Macrocycle-Profile)"

        calculated_profiles.append({
            'Compound_ID': comp_id,
            'Standardized_SMILES': smiles_str,
            'MW': round(mw, 2),
            'LogP': round(logp, 2),
            'HBD': hbd,
            'HBA': hba,
            'TPSA': round(tpsa, 2),
            'RotB': rotb,
            'Lipinski_Violations': lipinski_violations,
            'Veber_Violations': veber_violations,
            'Drug_Like': drug_like_status,
            'InChIKey_Hash': inchikey_hash
        })

    # Compile the final physical property baseframe
    df_physchem_master = pd.DataFrame(calculated_profiles)

    # Save the master physical profile to disk workspace
    df_physchem_master.to_csv(output_csv, index=False)

    # Calculate summary metrics for the user log
    passed_standard = len(df_physchem_master[df_physchem_master['Drug_Like'] == "Yes"])
    macrocyclic_profile = len(df_physchem_master[df_physchem_master['Drug_Like'] != "Yes"])

    print("✨ Physicochemical Profiling Complete!")
    print(f"📊 Total compounds profiled: {len(df_physchem_master)}")
    print(f"💊 Compounds matching traditional drug-like space: {passed_standard}")
    print(f"🧬 Macrocyclic structural variants flagged: {macrocyclic_profile}")
    print(f"🚀 Properties baseline directory saved to: '{output_csv}'")

    return df_physchem_master

# --- Run the Molecular Analytics Profiling Routine ---
df_properties_master = calculate_eribulin_library_properties()

# Display the interactive grid
df_properties_master

🧬 Loading standardized compound tracks from: 'Eribulin_Analogs_Standardized_Master_2D.csv'...
🧮 Calculating topological descriptors across 315 library leads...

✨ Physicochemical Profiling Complete!
📊 Total compounds profiled: 315
💊 Compounds matching traditional drug-like space: 114
🧬 Macrocyclic structural variants flagged: 201
🚀 Properties baseline directory saved to: 'Eribulin_Analogs_Physicochemical_Master.csv'


,Compound_ID,Standardized_SMILES,MW,LogP,HBD,HBA,TPSA,RotB,Lipinski_Violations,Veber_Violations,Drug_Like,InChIKey_Hash
0,CID_11354606,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
1,CID_118753181,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
2,CID_124604988,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
3,CID_6331630,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
4,CID_70679001,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,730.92,2.72,2,11,148.01,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
...,...,...,...,...,...,...,...,...,...,...,...,...
310,CID_175973608,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
311,CID_176489572,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed
312,CID_177442664,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,715.88,3.05,2,12,146.39,3,1,1,No (Macrocycle-Profile),Calculation-Bypassed
313,CID_177540367,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,729.91,3.44,2,12,146.39,4,1,1,No (Macrocycle-Profile),Calculation-Bypassed


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Enable interactive table display in Colab
data_table.enable_dataframe_formatter()

def run_eribulin_tautomer_canonicalization(input_csv="Eribulin_Analogs_Physicochemical_Master.csv", output_csv="Eribulin_Analogs_Physio_Master_2D.csv"):
    """
    Ingests the physicochemical profiling matrix, removes transient charges,
    and locks heterocycles and functional groups into their most stable canonical tautomer.
    """
    if not os.path.exists(input_csv):
        print(f"❌ Error: Baseline file '{input_csv}' not detected. Run Step 5 first!")
        return None

    print(f"🧬 Loading physicochemical screening dataset: '{input_csv}'...")
    df_properties = pd.read_csv(input_csv)

    # Initialize RDKit's high-fidelity standardization components
    uncharger = rdMolStandardize.Uncharger()
    tautomer_engine = rdMolStandardize.TautomerEnumerator()

    refined_records = []
    tautomer_shifts_detected = 0

    print(f"🔄 Normalizing proton equilibria and tautomeric states across {len(df_properties)} analogs...\n")

    for _, row in df_properties.iterrows():
        comp_id = row['Compound_ID']
        std_smiles = row['Standardized_SMILES']

        mol = Chem.MolFromSmiles(std_smiles)
        if not mol:
            continue

        try:
            # 1. Neutralize transient charges to achieve physiological equilibration
            equilibrated_mol = uncharger.uncharge(mol)

            # 2. Score and isolate the most stable canonical tautomeric form
            canonical_mol = tautomer_engine.Canonicalize(equilibrated_mol)
            refined_smiles = Chem.MolToSmiles(canonical_mol, isomericSmiles=True)

            # Track if a tautomeric proton shift or redraw took place
            if refined_smiles != std_smiles:
                tautomer_shifts_detected += 1

        except Exception:
            # Fall back to incoming standardized structure if parsing hits an exception
            refined_smiles = std_smiles

        # Generate a safe, verified structure signature token for downstream tracking
        try:
            inchikey = Chem.Inchi.MolToInchiKey(Chem.MolFromSmiles(refined_smiles))
            if not inchikey:
                inchikey = row['InChIKey_Hash']
        except Exception:
            inchikey = row['InChIKey_Hash']

        # Construct data columns preserving all calculated properties from Step 5
        refined_records.append({
            'Compound_ID': comp_id,
            'Standardized_SMILES': std_smiles,
            'Refined_SMILES': refined_smiles,
            'MW': row['MW'],
            'LogP': row['LogP'],
            'HBD': row['HBD'],
            'HBA': row['HBA'],
            'TPSA': row['TPSA'],
            'RotB': row['RotB'],
            'Drug_Like': row['Drug_Like'],
            'Refined_InChIKey': inchikey
        })

    # Compile final data frame
    df_physio_master = pd.DataFrame(refined_records)

    # Export the consolidated 2D master library to disk workspace
    df_physio_master.to_csv(output_csv, index=False)

    print("✨ Physiological Tautomer Canonicalization Complete!")
    print(f"📊 Total compounds processed: {len(df_physio_master)}")
    print(f"⚡ Shifts/proton redistributions resolved: {tautomer_shifts_detected}")
    print(f"🚀 Screening-ready 2D library locked and exported to: '{output_csv}'")

    return df_physio_master

# --- Run the Tautomeric Refinement Engine ---
df_tautomer_preview = run_eribulin_tautomer_canonicalization()

# Display the interactive comparison matrix
df_tautomer_preview[['Compound_ID', 'Standardized_SMILES', 'Refined_SMILES', 'Drug_Like', 'Refined_InChIKey']]

🧬 Loading physicochemical screening dataset: 'Eribulin_Analogs_Physicochemical_Master.csv'...
🔄 Normalizing proton equilibria and tautomeric states across 315 analogs...



[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Removed positive charge.
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Removed negative charge.
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Removed positive charge.
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Unc

✨ Physiological Tautomer Canonicalization Complete!
📊 Total compounds processed: 315
⚡ Shifts/proton redistributions resolved: 6
🚀 Screening-ready 2D library locked and exported to: 'Eribulin_Analogs_Physio_Master_2D.csv'


[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:47] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Running Uncharger
[23:35:48] Run

,Compound_ID,Standardized_SMILES,Refined_SMILES,Drug_Like,Refined_InChIKey
0,CID_11354606,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,No (Macrocycle-Profile),Calculation-Bypassed
1,CID_118753181,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,No (Macrocycle-Profile),Calculation-Bypassed
2,CID_124604988,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,No (Macrocycle-Profile),Calculation-Bypassed
3,CID_6331630,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,No (Macrocycle-Profile),Calculation-Bypassed
4,CID_70679001,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@H]6[C@@H](O3)[...,No (Macrocycle-Profile),Calculation-Bypassed
...,...,...,...,...,...
310,CID_175973608,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,C=C1CC2CCC34CC5OC6C(OC7CCC(CC(=O)CC8C(CC9OC(CC...,No (Macrocycle-Profile),Calculation-Bypassed
311,CID_176489572,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,C=C1C[C@H]2CC[C@]34C[C@H]5O[C@@H]6[C@@H](O3)[C...,No (Macrocycle-Profile),Calculation-Bypassed
312,CID_177442664,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,C=C1C[C@@H]2CC[C@@]34C[C@H]5O[C@@H]6[C@H](O3)[...,No (Macrocycle-Profile),Calculation-Bypassed
313,CID_177540367,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,C=C1C[C@@H]2CC[C@H]3C[C@@H](C)C(=C)[C@@H](C[C@...,No (Macrocycle-Profile),Calculation-Bypassed
